# decision tree — Python demo

Numerical companion to the entry [decision tree](https://dictionaryofml.org/terms/decisiontree.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

A single depth-2 regression tree is trained, in the CART fashion the entry describes, on actual weather measurements from GeoSphere Austria: the days of 2024 at the Krems an der Donau station (3805) that carry both readings, with the air temperature at 06:00 UTC as the feature and the temperature at 15:00 UTC of the same day as the label. Growing the tree greedily — each split chooses the threshold that most reduces the variance of the labels routed to the children — yields a piecewise constant hypothesis with four pieces, one per leaf node.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/decisiontree.py`](https://dictionaryofml.org/terms/decisiontree.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 decisiontree.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "decisiontree.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""decisiontree.py — numerical companion to the glossary entry
'decisiontree' (decision tree).

A single depth-2 regression tree is trained, in the CART fashion the
entry describes, on actual weather measurements from GeoSphere Austria:
the days of 2024 at the Krems an der Donau station (3805) that carry
both readings, with the air temperature at 06:00 UTC as the feature and
the temperature at 15:00 UTC of the same day as the label.  Growing the tree greedily — each split chooses the threshold
that most reduces the variance of the labels routed to the children —
yields a piecewise constant hypothesis with four pieces, one per leaf
node.

Blocks
------
[B-fetch]   Download the hourly record of 2024 from GeoSphere, keep the
            days carrying both readings, and write them to
            decisiontree_weather.csv; check the count and one pinned
            measurement.
[B-tree]    Grow a depth-2 regression tree by greedy variance
            splitting; check that the hypothesis is piecewise constant
            with at most four pieces, that each split reduced the
            variance, and that the tree predicts the label far better
            than a constant.
[B-picture] Write the scatterplot and the step-function curve read by
            the entry's figure (decisiontree_scatter.csv,
            decisiontree_curve.csv).

Deterministic: historical measurements, greedy splits, no randomness.
Self-contained: numpy + matplotlib only (stdlib urllib for the
download).

Outputs
-------
decisiontree_weather.csv : day, t06, t15 for the 365 days of 2024
                           that carry both readings
decisiontree_scatter.csv : t06, t15 of those days
decisiontree_scatter_plot.csv : every 19th of those days, the 20 circles of the figure
decisiontree_curve.csv   : t06, pred -- the tree's step function
decisiontree.png         : preview (checking only) -- the scatterplot
                           with the depth-2 tree's step function
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-fetch]** Download the hourly record of 2024 from GeoSphere, keep the days carrying both readings, and write them to decisiontree_weather.csv; check the count and one pinned measurement.

In [ ]:
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1h?parameters=tl&station_ids=3805"
       "&start=2024-01-01T00:00&end=2024-12-31T23:00")
with urllib.request.urlopen(URL, timeout=300) as resp:
    payload = json.load(resp)
readings = payload["features"][0]["properties"]["parameters"]["tl"]["data"]
rows = {}
for stamp, value in zip(payload["timestamps"], readings):
    if value is None:                      # the archive drops whole days
        continue
    hour = stamp[11:16]                    # the stamps are UTC, and so are
    if hour in ("06:00", "15:00"):         # the two times, so no conversion
        rows.setdefault(stamp[:10], {})[hour] = float(value)
days = [d for d in sorted(rows) if len(rows[d]) == 2]
t06 = np.array([rows[d]["06:00"] for d in days])
t15 = np.array([rows[d]["15:00"] for d in days])
with open(OUT_DIR / "decisiontree_weather.csv", "w") as f:
    f.write("day,t06,t15\n")
    for d in days:
        f.write(f"{d},{rows[d]['06:00']:.1f},{rows[d]['15:00']:.1f}\n")
check("[B-fetch] 365 of the days of 2024 carry both readings (the hourly "
      "record drops 2024-01-12)", len(days) == 365)
check("[B-fetch] the record matches the archive "
      "(2024-05-02: 16.1 at 06:00 and 22.2 at 15:00, both UTC)",
      rows["2024-05-02"]["06:00"] == 16.1
      and rows["2024-05-02"]["15:00"] == 22.2)

**[B-tree]** Grow a depth-2 regression tree by greedy variance splitting; check that the hypothesis is piecewise constant with at most four pieces, that each split reduced the variance, and that the tree predicts the label far better than a constant.

In [ ]:
MINLEAF = 20


def best_split(x, y):
    """Threshold minimizing the summed squared error of the two parts."""
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    best, best_sse = None, np.inf
    for i in range(MINLEAF, len(xs) - MINLEAF + 1):
        if xs[i - 1] == xs[i]:
            continue
        left, right = ys[:i], ys[i:]
        sse = ((left - left.mean()) ** 2).sum() \
            + ((right - right.mean()) ** 2).sum()
        if sse < best_sse:
            best_sse, best = sse, (xs[i - 1] + xs[i]) / 2
    return best, best_sse


root_t, root_sse = best_split(t06, t15)
cuts = [root_t]
for side in (t06 <= root_t, t06 > root_t):
    t, _ = best_split(t06[side], t15[side])
    if t is not None:
        cuts.append(t)
cuts = sorted(cuts)
edges = [-np.inf] + cuts + [np.inf]
means = np.array([t15[(t06 > lo) & (t06 <= hi)].mean()
                  for lo, hi in zip(edges[:-1], edges[1:])])


def predict(xq):
    return means[np.searchsorted(np.array(cuts), xq)]


err_tree = float(np.mean((t15 - predict(t06)) ** 2))
err_const = float(np.var(t15))
print(f"    thresholds {[round(c, 2) for c in cuts]}, "
      f"leaf values {[round(m, 1) for m in means]}")
print(f"    average squared error: tree {err_tree:.1f}, "
      f"constant {err_const:.1f}")
check("[B-tree] the hypothesis is piecewise constant with at most "
      "four pieces", len(means) <= 4)
check("[B-tree] the root split reduced the variance of the labels",
      root_sse < err_const * len(t15))
check("[B-tree] the tree predicts the label far better than a constant",
      err_tree < err_const / 3)

**[B-picture]** Write the scatterplot and the step-function curve read by the entry's figure (decisiontree_scatter.csv, decisiontree_curve.csv).

In [ ]:
np.savetxt(OUT_DIR / "decisiontree_scatter.csv",
           np.stack([t06, t15], 1),
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
# the entry's figure draws every 19th day, 20 of the 366, so the open
# circles do not hide the step function; the tree is learned from all 366
PLOT_EVERY = 19
np.savetxt(OUT_DIR / "decisiontree_scatter_plot.csv",
           np.stack([t06[::PLOT_EVERY], t15[::PLOT_EVERY]], 1),
           delimiter=",", header="t06,t15", comments="", fmt="%.1f")
check("[B-picture] the plotted subset holds 20 days", t06[::PLOT_EVERY].size == 20)
bounds = [t06.min() - 0.5] + cuts + [t06.max() + 0.5]
gx, gy = [], []
for k, (lo, hi) in enumerate(zip(bounds[:-1], bounds[1:])):
    gx += [lo, hi]
    gy += [means[k], means[k]]
gx, gy = np.array(gx), np.array(gy)
np.savetxt(OUT_DIR / "decisiontree_curve.csv",
           np.stack([gx, gy], 1),
           delimiter=",", header="t06,pred", comments="", fmt="%.2f")
check("[B-picture] the written curve has one constant level per leaf",
      len(set(np.round(gy, 4))) == len(means))

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(t06, t15, "o", ms=2.5, color="0.6", label="days of 2024")
ax.plot(gx, gy, "-", lw=2.2, color="black",
        label="depth-2 tree (step function)")
ax.set_xlabel("minimum temperature of the day (°C)")
ax.set_ylabel("maximum temperature of the day (°C)")
ax.set_title("a depth-2 regression tree at Kaisaniemi")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "decisiontree.png", dpi=150)

failed = [name for name, ok in report if not ok]
print(f"{len(report) - len(failed)}/{len(report)} checks passed"
      + (f", FAILED: {failed}" if failed else ""))